In [0]:
%python
# Databricks notebook source
# MAGIC %python

from datetime import datetime
import re

BUCKET = "s3://sneha-retail-sales-data-warehouse"
zones = ["sftp", "raw"]
entities = ["customers", "products", "stores", "sales"]


def extract_datetime(filename):
    m = re.search(r'_(\d{14})\.csv$', filename)
    if not m:
        return datetime.min

    ts = m.group(1)   # DDMMYYYYHHMMSS
    return datetime.strptime(ts, "%d%m%Y%H%M%S")


for zone in zones:
    for entity in entities:

        path = f"{BUCKET}/{zone}/{entity}/"
        archive_path = f"{BUCKET}/archive/{zone}/{entity}/"

        try:
            files = [f for f in dbutils.fs.ls(path) if f.name.endswith(".csv")]

            if len(files) <= 1:
                print(f"No archive needed → {zone}/{entity}")
                continue

            files_sorted = sorted(
                files,
                key=lambda x: extract_datetime(x.name),
                reverse=True
            )

            latest = files_sorted[0]
            old_files = files_sorted[1:]

            print(f"\nKeeping latest: {latest.name}")

            for f in old_files:
                dbutils.fs.mv(f.path, archive_path + f.name)
                print(f"Archived → {f.name}")

        except Exception as e:
            print(f"Skipped → {zone}/{entity}: {e}")